# vLLM Benchmark

Run the cell below. First run starts the vLLM server, waits for it to load, then benchmarks. Re-running skips straight to the benchmark (server is reused).

In [ ]:
%%bash
PORT=8010
ENDPOINT=http://localhost:$PORT

# If already up, say so and stop
if curl -sf $ENDPOINT/v1/models >/dev/null 2>&1; then
  echo "vLLM already running on $PORT — ready."
  exit 0
fi

echo "Starting vLLM..."
VLLM_ATTENTION_BACKEND=TRITON_ATTN nohup vllm serve Qwen/Qwen2.5-7B-Instruct-AWQ --served-model-name Qwen/Qwen2.5-7B-Instruct-AWQ \
  --port $PORT --tensor-parallel-size 1 --dtype float16 \
  --gpu-memory-utilization 0.80 --max-model-len 32768 --max-num-seqs 64 \
  --enable-prefix-caching --trust-remote-code --disable-custom-all-reduce \
  > /tmp/vllm.log 2>&1 &

for i in $(seq 1 120); do
  curl -sf $ENDPOINT/v1/models >/dev/null 2>&1 && break
  echo "waiting... ($((i*10))s)"; sleep 10
done

if ! curl -sf $ENDPOINT/v1/models >/dev/null 2>&1; then
  echo "vLLM failed to start:"; tail -n 40 /tmp/vllm.log; exit 1
fi

echo "Ready for Benchmarking..."